## Ripple data analysis

Contains Ripple data collected from Gian over two sessions.


In [1]:
# Loading/Import related packages
import sys
import os

#Usual suspects
import pandas as pd
import numpy as np
import json
import pickle as pkl
import matplotlib.pyplot as plt

#Extras for plotting
from matplotlib.patches import Patch

#Extra for typing
from collections import defaultdict

# Needed to point the importer to the src folder
sys.path.insert(0,'/Users/lizkal/Library/CloudStorage/SynologyDrive-Personal/MotorUnitSuite')

#Decomposition/Processing Imports
from src.muniverse.algorithms.cbss import CBSS

In [8]:
REPO_DIR  = os.path.abspath(os.getcwd())
INPUT_DIR = os.path.join(REPO_DIR, 'data')
OUTPUT_DIR = os.path.join(REPO_DIR, 'results')

filename = 'emg_recording_giuan_tewst_20260826_122524.pkl'



In [9]:
def load_simulation_data(input_dir, filename):

    """
    Load the simulation data from a pickle file.

    Parameters:
    - input_dir: str, the directory where the pickle file is located.
    - filename: str, the name of the pickle file.

    Returns:
    - emg_array: dict, the loaded simulation EMG data. 
    Normally contains 320 channels, here downsampled to the first 8x8 array to make decomposition faster .
    """
    file_path = os.path.join(input_dir, filename)

    # Check if the file exists before attempting to load it
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist. Please ensure the input directory and filename are correct.")

    result_pandas = pd.read_pickle(file_path)
    print("Loaded data from:", file_path)
    print(result_pandas.keys())
    print(result_pandas['emg'].shape)

    #The array generated by the simulation is a 320 channel array, split between several 8x8 arrays.
    #but for the sake of speed, we will only use the first 8x8 array for decomposition

    #The data is formatted as n_rows x n_cols x n_samples, so we can slice it to get the first 8x8 array

    emg_array = result_pandas['emg'][0:8, 0:8, :]

    #These two arrays contain the intended poses and durations for each pose
    # We will use this information later in the analysis, see below in the Decomposition Section
    poses = result_pandas['metadata']['poses_intended']
    durations = result_pandas['metadata']['durations']


    #NOTE: a tuple, not a set: a set has no order, so the unpacking below would be random
    return emg_array, poses, durations
    

In [10]:
emg_array, poses, durations = load_simulation_data(input_dir=INPUT_DIR, filename=filename)

Loaded data from: /Users/lizkal/Library/CloudStorage/SynologyDrive-Personal/Ripple/data/emg_recording_giuan_tewst_20260826_122524.pkl
dict_keys(['data', 'srate', 'n_channels', 'filtered', 'pre_trigger_seconds', 'timestamp', 'stream_type', 'trial_metadata', 'session_timeline'])


KeyError: 'emg'

In [12]:
file_path = os.path.join(INPUT_DIR, filename)
result_pandas = pd.read_pickle(file_path)

print(result_pandas.keys())

dict_keys(['data', 'srate', 'n_channels', 'filtered', 'pre_trigger_seconds', 'timestamp', 'stream_type', 'trial_metadata', 'session_timeline'])


In [14]:
print(result_pandas['trial_metadata'])

{'appName': 'patientgui-frontend', 'appVersion': '0.1.0', 'gitSha': '4eb78f80f9f5e16bc73a02bc11951bc791349955-dirty', 'subjectId': 'giuan', 'sessionId': 'tewst', 'notes': '', 'sequenceName': 'gian-ftfe', 'totalItems': 3, 'settings': {'pauseBetweenItems': 20, 'restBetweenReps': 7, 'prepTime': 3, 'playbackSpeed': 1}, 'items': [{'index': 0, 'model': 'fist.glb', 'animation': 0, 'repetitions': 8}, {'index': 1, 'model': 'fasttripodpinch.glb', 'animation': 0, 'repetitions': 8}, {'index': 2, 'model': 'fastfingerext.glb', 'animation': 0, 'repetitions': 8}], 'startTime': '2026-08-26T10:25:24.579Z'}


In [ ]:
print(result_pandas['data'].shape)

In [ ]:
print(result_pandas['session_timeline'])